In [1]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"
from joblib import Parallel, delayed

import numpy as np

from dysts.maps import Henon

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error

from spring_rc.v1 import (
    build_von_mises_lattice,
    run_simulation,
    scatter_plot,
    save_figure,
    r_2_plots_grid,
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Henon

In [2]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [3]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

# Single

In [4]:
rng = np.random.default_rng(42)

num_cells = 100
cell_spacing = 2.5
wall_dist = 1.0
mass_val = 0.01
damping_val = 0.2
k_wall_val = 5.0
k_inter_val = 4.0
rest_wall_ratio = 1.1
rest_inter_ratio = 1.1
k_wall_spread = 1.0
k_inter_spread = 1.0
rest_wall_ratio_spread = 0.0
rest_inter_ratio_spread = 0.0

dt = 0.01
target_node_count = 10
input_force = 10
free_steps = 3

N_step = int(num_cells / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3

F_in = np.zeros((total_steps_after_chaos, target_node_count, 2))
for j in range(target_nodes.shape[0]):
    F_in[::, j, 0] = henon_scaled[:, j % 2]
F_in = F_in * input_force

In [5]:
(
    init_pos,
    m_inv_diag,
    c_diag,
    connections_list,
    wall_nodes,
    k_vals,
    rest_lens,
) = build_von_mises_lattice(
    rng=rng,
    num_cells=num_cells,
    cell_spacing=cell_spacing,
    wall_dist=wall_dist,
    mass_val=mass_val,
    damping_val=damping_val,
    k_wall_val=k_wall_val,
    k_wall_spread=k_wall_spread,
    k_inter_val=k_inter_val,
    k_inter_spread=k_inter_spread,
    rest_wall_ratio=rest_wall_ratio,
    rest_wall_ratio_spread=rest_wall_ratio_spread,
    rest_inter_ratio=rest_inter_ratio,
    rest_inter_ratio_spread=rest_inter_ratio_spread,
)

In [6]:
Xd, _ = run_simulation(
    steps=total_steps_after_chaos,
    free_steps=free_steps,
    dt=dt,
    m_inv=m_inv_diag,
    c_diag=c_diag,
    F_in=F_in,
    target_nodes=target_nodes,
    init_pos=init_pos,
    conn=connections_list,
    k=k_vals,
    rest=rest_lens,
    fixed_nodes=wall_nodes,
    rec_nodes=1 + 3 * np.arange(num_cells),
)


In [10]:
X_data = Xd[transient_steps_reservoir:-tau_steps]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
x_scaler = StandardScaler()
X_train, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)
Y_train, Y_test_unscaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)
model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)
Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

In [11]:
rmse = root_mean_squared_error(Y_test, Y_pred, multioutput="raw_values")
nrmse = rmse / np.std(Y_test, axis=0)
r_2 = r2_score(Y_test, Y_pred, multioutput="raw_values")
nrmse, r_2

(array([0.84325412, 0.02982294]), array([0.28892249, 0.99911059]))

# Non Dim